In [2]:
from gerrychain import Graph, Election, Partition, MarkovChain, constraints
from gerrychain.updaters import cut_edges, Tally
from gerrychain.accept import always_accept
from gerrychain.proposals import recom
import matplotlib.pyplot as plt
from functools import partial
import pandas as pd
import numpy as np

In [ ]:
graph_pa = Graph.from_file("./PA/PA.shp")

In [ ]:
# Total population in the graph
tot_pop = sum(graph_pa.nodes[v]['TOTPOP'] for v in graph_pa.nodes())
tot_pop

13002674.0

In [ ]:
elections = {
    Election("PRES24", {"Democratic" : "G24PREDHAR", "Republican" : "G24PRERTRU"}),
    Election("GEN24", {"Democratic" : "G24USSDCAS", "Republican" : "G24USSRMCC"}),
    Election("ATG24", {"Democratic" : "G24ATGDDEP", "Republican" : "G24ATGRSUN"}) 
}

In [ ]:
my_updaters = {
    "cut_edges" : cut_edges,
    "population": Tally("TOTPOP", alias="population"),
    "hispanic_population": Tally("HISP", alias="hispanic_population"),
    "black_population": Tally("NH_BLACK", alias="black_population")
}

election_updaters = {
    election.name: election for election in elections
}

my_updaters.update(election_updaters)

In [ ]:
initial_partition = Partition(
    graph_pa,
    assignment = "CD",
    updaters = my_updaters
)

In [ ]:
# Define the 2024 presidential election
pres24 = Election("PRES24", {"Democratic" : "G24PREDHAR", "Republican" : "G24PRERTRU"})

# Create the starting partition
initial_partition = Partition(
    graph_pa,
    assignment = "CD",
    updaters = {
        "cut_edges" : cut_edges,
        "population": Tally("TOTPOP", alias="population"),
        "hispanic_population": Tally("HISP", alias="hispanic_population"),
        "black_population": Tally("NH_BLACK", alias="black_population"),
        "PRES24": pres24
    }    
)

initial_partition

<Partition [17 parts]>

In [ ]:
def build_partition(election_name, election):
    partition = Partition(
        graph_pa,
        assignment = "CD",
        updaters = {
            "cut_edges" : cut_edges,
            "population": Tally("TOTPOP", alias="population"),
            "hispanic_population": Tally("HISP", alias="hispanic_population"),
            "black_population": Tally("NH_BLACK", alias="black_population"),
            election_name : election
        }    
    )

    return partition

In [ ]:
num_dist = 17
ideal_pop = tot_pop/num_dist
pop_tolerance = 0.03

# Set up the ReCom proposal for the random walk
random_walk_prop = partial(
    recom, 
    pop_col="TOTPOP",
    pop_target = ideal_pop,
    epsilon= pop_tolerance,
    node_repeats = 2
)

In [ ]:
# Delete soon
# Creating the population balance contraint
population_constraint = constraints.within_percent_of_ideal_population(
    initial_partition, 
    pop_tolerance, 
    pop_key = "population"
)

In [7]:
# Building Markov chain
random_walk_20 = MarkovChain(
    proposal = random_walk_prop,
    constraints = [population_constraint], 
    accept = always_accept,
    initial_state = initial_partition,
    total_steps = 20000
)

random_walk_40 = MarkovChain(
    proposal = random_walk_prop,
    constraints = [population_constraint], 
    accept = always_accept,
    initial_state = initial_partition,
    total_steps = 40000
)

In [ ]:
def build_chain(steps):

    population_constraint = constraints.within_percent_of_ideal_population(
        initial_partition, 
        pop_tolerance, 
        pop_key = "population"
    )

    chain = MarkovChain(
        proposal = random_walk_prop,
        constraints = [population_constraint], 
        accept = always_accept,
        initial_state = initial_partition,
        total_steps = steps
    )

    return chain

In [ ]:
random_walk_20 = build_chain(20000)
random_walk_40 = build_chain(40000)

In [ ]:
# Run the chain and collect the edge counts
cut_edges_list_20 = []
efficiency_gap_list_20 = []
dem_wins_list = []
dem_vote_shares = []

black_majority = []

for part in random_walk_20:

    pres_eg = part["PRES24"].
    cut_edges_list_20.append(len(part["cut_edges"]))
    
    efficiency_gap_list_20.append(part["PRES24"].efficiency_gap())
    dem_wins_list.append(part["PRES24"].wins("Democratic"))
    dem_vote_shares.append(sorted(part["PRES24"].percents("Democratic")))

    count = 0

    # Do the same for the other populations
    for district in part.parts:
        black_pop = part["black_population"][district]
        total_pop = part["population"][district]

        if black_pop / total_pop > 0.5:
            count += 1

    black_majority.append(count)

print(len(cut_edges_list_20))

In [8]:
# Run the chain and collect the edge counts
cut_edges_list_20 = []
efficiency_gap_list_20 = []
dem_wins_list = []
dem_vote_shares = []

black_majority = []

for part in random_walk_20:
    cut_edges_list_20.append(len(part["cut_edges"]))
    efficiency_gap_list_20.append(part["PRES24"].efficiency_gap())
    dem_wins_list.append(part["PRES24"].wins("Democratic"))
    dem_vote_shares.append(sorted(part["PRES24"].percents("Democratic")))

    count = 0

    # Do the same for the other populations
    for district in part.parts:
        black_pop = part["black_population"][district]
        total_pop = part["population"][district]

        if black_pop / total_pop > 0.5:
            count += 1

    black_majority.append(count)

print(len(cut_edges_list_20))

c:\Users\dcviv\miniconda3\envs\vini\Lib\site-packages\gerrychain\tree.py:704: BipartitionWarning: 
Failed to find a balanced cut after 1000 attempts.
If possible, consider enabling pair reselection within your
MarkovChain proposal method to allow the algorithm to select
a different pair of districts for recombination.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
cut_edges_list_40 = []
efficiency_gap_list_40 = []
dem_wins_list_40 = []
dem_vote_shares_40 = []

black_majority_40 = []

for part in random_walk_40:
    cut_edges_list_40.append(len(part["cut_edges"]))
    efficiency_gap_list_40.append(part["PRES24"].efficiency_gap())
    dem_wins_list_40.append(part["PRES24"].wins("Democratic"))
    dem_vote_shares_40.append(sorted(part["PRES24"].percents("Democratic")))

    count = 0

    # Do the same for the other populations
    for district in part.parts:
        black_pop = part["black_population"][district]
        total_pop = part["population"][district]

        if black_pop / total_pop > 0.5:
            count += 1

    black_majority_40.append(count)


print(len(cut_edges_list_40))

In [ ]:
data = pd.DataFrame(dem_vote_shares)

fig, ax = plt.subplots(figsize=(10, 6))

# 50% line
ax.axhline(0.5, color="#cccccc")

data.boxplot(ax=ax, positions=range(len(data.columns)))

# Drawing the Democratic vote share percentage (first row [0])
current_plan = data.iloc[0]
ax.plot(range(len(current_plan)), current_plan, "ro")

ax.set_ylim(0.25, 1)

ax.set_title("Comparing the 2022 PA Congressional Plan to an Ensemble")
ax.set_ylabel("Democratic vote share (President 2024)")
ax.set_xlabel("Sorted districts")

plt.show()

In [ ]:
# Plotting the the cut edge counts
plt.figure(figsize=(12, 6))

plt.hist(cut_edges_list_20, align='left')

plt.show()


In [ ]:
plt.figure(figsize=(10, 6))

plt.hist(efficiency_gap_list_20, bins=30)
plt.axvline(0, color="black", linestyle="--")

plt.title("Efficiency Gap Distribution: 2024 Presidential")

plt.xlabel("Efficiency Gap")
plt.ylabel("Frequency")

plt.show()

In [ ]:
# Plotting the the cut edge counts
plt.figure(figsize=(12, 6))

plt.hist(cut_edges_list_40, align='left')

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.hist(efficiency_gap_list_40, bins=30)
plt.axvline(0, color="black", linestyle="--")

plt.title("Efficiency Gap Distribution: 2024 Presidential")

plt.xlabel("Efficiency Gap")
plt.ylabel("Frequency")

plt.show()

In [ ]:
# Define the 2024 General Election (Senate)
sen24 = Election("GEN24", {"Democratic" : "G24USSDCAS", "Republican" : "G24USSRMCC"})

# Create the starting partition
initial_partition2 = Partition(
    graph_pa,
    assignment = "CD",
    updaters = {
        "cut_edges" : cut_edges,
        "population": Tally("TOTPOP", alias="population"),
        "hispanic_population": Tally("HISP", alias="hispanic_population"),
        "black_population": Tally("NH_BLACK", alias="black_population"),
        "GEN24": sen24
    }    
)

initial_partition2

In [ ]:
# Building a short Markov chain
random_walk2 = MarkovChain(
    proposal = random_walk_prop,
    constraints = [population_constraint],
    accept = always_accept,
    initial_state = initial_partition2,
    total_steps = 20000
)

In [ ]:
# Run the chain and collect the edge counts
cut_edges_list2 = []

for part in random_walk2:
    cut_edges_list2.append(len(part["cut_edges"]))

print(cut_edges_list2)

In [ ]:
# Plotting the the cut edge counts for the senate election
plt.figure(figsize=(12, 6))

plt.hist(cut_edges_list2, align='left')

plt.show()

In [ ]:
# Define the 2024 General Election (Attorney General)
atg24 = Election("ATG24", {"Democratic" : "G24ATGDDEP", "Republican" : "G24ATGRSUN"}) 

# Create the starting partition
initial_partition3 = Partition(
    graph_pa,
    assignment = "CD",
    updaters = {
        "cut_edges" : cut_edges,
        "population": Tally("TOTPOP", alias="population"),
        "hispanic_population": Tally("HISP", alias="hispanic_population"),
        "black_population": Tally("NH_BLACK", alias="black_population"),
        "ATG24": atg24
    }    
)

initial_partition3

In [ ]:
# Building a short Markov chain
random_walk3 = MarkovChain(
    proposal = random_walk_prop,
    constraints = [population_constraint],
    accept = always_accept,
    initial_state = initial_partition3,
    total_steps = 20000
)

In [ ]:
# Run the chain and collect the edge count
cut_edges_list3 = []

for part in random_walk3:
    cut_edges_list3.append(len(part["cut_edges"]))

print(cut_edges_list3)

In [ ]:
# Plotting the the cut edge counts for attorney general election
plt.figure(figsize=(12, 6))

plt.hist(cut_edges_list3, align='left')

plt.show()

In [ ]:
def chain_results(chain, elections, party="Democratic"):
    results = {}

    for election in elections:
        results[election] = {
            "cut_edges": [],
            "efficiency_gap": [],
            "party_wins": [],
            "party_vote_shares": [],
        }

    results["population"] = {
        "black_majority": [],
        "hispanic_majority": [],
        "max_black_share": [],
        "max_hispanic_share": []
    }

    for part in chain:

        for election in elections:
            results[election]["cut_edges"].append(len(part["cut_edges"]))
            results[election]["efficiency_gap"].append(part[election].efficiency_gap())
            results[election]["party_wins"].append(part[election].wins(party))
            results[election]["party_vote_shares"].append(sorted(part[election].percents(party)))

        black_majority_count = 0
        hispanic_majority_count = 0
        black_shares = []
        hispanic_shares = []

        for district in part.parts:
            total_pop = part["population"][district]

            black_share = part["black_population"][district] / total_pop
            hispanic_share = part["hispanic_population"][district] / total_pop

            black_shares.append(black_share)
            hispanic_shares.append(hispanic_share)

            if black_share > 0.5:
                black_majority_count += 1

            if hispanic_share > 0.5:
                hispanic_majority_count += 1

        results["population"]["black_majority"].append(black_majority_count)
        results["population"]["hispanic_majority"].append(hispanic_majority_count)
        results["population"]["max_black_share"].append(max(black_shares))
        results["population"]["max_hispanic_share"].append(max(hispanic_shares))


    return results

In [ ]:
election_names = ["PRES24", "SEN24", "ATG24"]

results_20 = chain_results(random_walk_20, election_names)
results_40 = chain_results(random_walk_40, election_names)

In [ ]:
vote_shares = results_20["PRES24"]["party_vote_shares"]

data = pd.DataFrame(vote_shares)

fig, ax = plt.subplots(figsize=(10, 6))

# 50% line
ax.axhline(0.5, color="#cccccc")

data.boxplot(ax=ax, positions=range(len(data.columns)))

# Drawing the Democratic vote share percentage (first row [0])
current_plan = data.iloc[0]
ax.plot(range(len(current_plan)), current_plan, "ro")

ax.set_ylim(0.25, 1)

ax.set_title("Comparing the 2022 PA Congressional Plan to an Ensemble")
ax.set_ylabel("Democratic vote share (President 2024)")
ax.set_xlabel("Sorted districts")

plt.show()